# Choice logit: model values for the psychometrics

**Question.** Meg's psychometrics use two values: the *true* magnitudes the task offers, and the
*believed* ones (what an arm paid the last time the mouse chose it). Neither is what the mouse
actually uses. This notebook estimates a third, **model value**, from the mouse's own choices, and
redraws the psychometrics and the stay/switch curves with it.

1. **The model.** A logistic regression of choice on port location (a fixed bias per tower),
   reward magnitude history and choice history.
2. **Synthetic data.** Simulated Q-learning mice (same task simulator and value learner as
   `scripts/to_integrate/RL model.ipynb`) whose real values and choice rule we know, and a
   **value-blind control** that repeats its choices and ignores rewards.
3. **Real data.** The same pipeline on the drifting-task cohorts, next to the believed-value
   versions of the figures.

**The trap this notebook is built around.** A model value made of recent rewards is non-zero mostly
for arms the mouse chose recently, and a mouse that repeats its choices picks them again. Ranking
the arms by such a value predicts choice even if the mouse ignores rewards entirely. So **every
curve below comes with a dashed baseline**: what a model with choice history and tower bias but *no
reward term* predicts for the same outcome on the same trials. Only the gap between solid and
dashed is evidence for a value effect. Section 2.5 checks this on an agent that is known to ignore
value.

## 0. Setup and parameters

Everything below is a call into `src/` (`src/pipelines/choice_logit_values.py` and the modules it
uses); the notebook only chooses parameters, calls them, and explains the result.

Real data are loaded **cache-first** from the import cache (`BANDIT_CACHE_DIR`): the newest cached
snapshot of each cohort is used, nothing is written to the data share, and sessions added since
the snapshot are missing (a `StaleCacheWarning` says so). Set `DATA_ROOT` only if the data are not
under `BANDIT_DATA_ROOT`. Figures go to `results/` (git-ignored) as pdf, png and svg with a
`.caption.txt` beside each.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/behavior_import").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.behavior_analysis import choice_logit_cv as cv
from src.behavior_analysis import model_value_psychometrics as mvp
from src.behavior_analysis.drifting_trial_table import drifting_trial_table
from src.behavior_visualization import plot_choice_logit as plots
from src.behavior_visualization import plot_null_agents as null_plots
from src.agents import null_agents
from src.pipelines import null_agents as nulls
from src.pipelines import choice_logit_values as pipe
from src.pipelines import regression_extensions as ext
from src.behavior_visualization import plot_regression_extensions as ext_plots
from src.settings import get_unchecked_settings


def show(base):
    # figures are saved (pdf, png, svg, caption) by the plot functions; display the png here
    display(Image(filename=f"{base}.png"))

In [ ]:
# ---- parameters (edit here) ------------------------------------------------------------
COHORTS = [1, 3]  # cohorts whose drifting problems are pooled in the real-data section
DATA_ROOT = None  # None: BANDIT_DATA_ROOT. Loading is cache-first and never writes to the share.
OUTPUT_DIR = REPO_ROOT / "results" / "figures" / "choice-logit-values"

N_LAGS = 5  # reward lags and choice lags in the model
TAIL_GRID = pipe.TAIL_GRID  # decays of the reward tail tried (chosen by held-out likelihood)
L2 = 1.0  # penalty on every weight (a Gaussian prior with SD 1 logit)
N_BINS = 5  # per-mouse quantile bins of the continuous value in the psychometrics
MIN_MICE = mvp.MIN_MICE  # a mean from fewer mice is not drawn
POOL_TRIALS = True  # within a mouse, pool the trials of a bin (False: average cells equally)

# Synthetic mice: (learning_rate, beta, kappa = stickiness, plus_bias = bias to plus-shaped ports)
PARAMETER_SETS = [
    (0.30, 1.0, 0.0, 0.0),  # plain Q-learner
    (0.30, 1.0, 1.0, 0.0),  # sticky
    (0.30, 1.0, 0.0, 0.8),  # tower (plus-port) bias
    (0.15, 1.2, 0.5, 0.5),  # slow learner, mild stickiness and bias
    (0.50, 0.8, 1.0, 0.5),  # fast learner, sticky and biased
]
MICE_PER_SET = 2
SYNTHETIC = dict(n_problems=6, sessions_per_problem=4, trials_per_session=200)
SEED = 20261106

# Negative control (section 2.5): ignores reward (beta = 0) but repeats its choices, next to a
# value-driven learner of the same learning rate; three mice of each.
CONTROL_SETS = {"Value-Blind Sticky": (0.30, 0.0, 2.5, 0.0), "Value-Driven": (0.30, 2.0, 1.0, 0.0)}

# Negative controls (sections 2.7 and 3.3): win-stay/lose-shift agents fitted to each mouse.
# The history trace of the agents decays by NULL_DECAY per trial (a half-life of about 7 trials).
NULL_DECAY = null_agents.DEFAULT_DECAY
NULL_SEED = SEED + 2

# Regression extensions (sections 2.8 and 3.4): every model variant is refitted on each of a
# mouse's problems, about 7 s per mouse; mice run in parallel and each mouse's result is cached.
EXT_JOBS = 2  # parallel workers over mice
EXT_BINS = 10  # quantile bins of the calibration figure (deciles)
EXT_CACHE_DIR = get_unchecked_settings().cache_dir / "regression_extensions"  # disposable

synthetic_dir, real_dir = OUTPUT_DIR / "synthetic", OUTPUT_DIR / "real"

## 1. The model

On every trial the mouse chooses one of the three offered ports $a$. Each gets a utility

$$
U_a(t) \;=\; \underbrace{b_{\text{tower}(a)}}_{\text{port location}}
\;+\; \underbrace{\sum_{k=1}^{K} w^{r}_k\, r_a(t-k) + w^{\text{tail}}\,\tau_a(t)}_{\text{reward history}}
\;+\; \underbrace{\sum_{k=1}^{K} w^{c}_k\, c_a(t-k)}_{\text{choice history}},
\qquad P(a)=\frac{e^{U_a}}{\sum_{a'} e^{U_{a'}}}
$$

- $r_a(t-k)$ is the reward magnitude (pulses, 0-4) the mouse **received** at port $a$ on trial $t-k$ if
  it chose $a$ then, otherwise 0.
- $\tau_a(t)=\sum_{j>K}\gamma^{\,j-K-1}\,r_a(t-j)$ is the **reward tail**: all rewards older than the
  $K$ lags, decayed by $\gamma$ per trial and summed into one regressor (one extra weight). Without
  it, an arm not chosen for $K$ trials has a model value of exactly 0 although the mouse still has
  a memory of it (a Q-learner keeps an unchosen arm's value indefinitely). $\gamma$ is chosen from
  `TAIL_GRID` by held-out likelihood (section 2.1).
- $c_a(t-k)$ is 1 if $a$ was chosen on trial $t-k$, otherwise 0. A positive $w^c_1$ is a tendency
  to repeat (perseveration).
- $b_{\text{tower}}$ is one fixed bias per tower (the 3x3 field has nine). Adding the same number to
  every tower's bias changes no probability, so the alphabetically first tower is fixed at 0.

It is a *conditional logit*: one set of weights per mouse, a softmax over the three offered ports,
fitted by maximum likelihood with a small L2 penalty (`L2`; a Gaussian prior with SD 1 logit on every
weight, weak next to thousands of trials) and L-BFGS-B with an exact gradient. A fit that does not
converge raises. The tower biases are only comparable if the problems share towers; the design warns
if the problems fall into groups that share none.

**History rules.** Reward history (lags and tail) carries over between the sessions of a problem and
starts empty at a problem's start, like the values of the Q-learner. Choice history is cleared at a
session start and after a dropped trial, as that learner's stickiness is. A lag that reaches back
before the start of the history is zero.

**Model value.** Only the reward part is called value:

$$\hat V_a(t) \;=\; \sum_{k=1}^{K} w^{r}_k\, r_a(t-k) + w^{\text{tail}}\,\tau_a(t).$$

Perseveration (choice weights) and port preference (tower biases) are *nuisance* terms: they are in
$U$, so the fit does not credit the reward kernel for them, but they are left out of $\hat V$.

**Why held-out.** For each problem of a mouse the weights come from a fit to the mouse's **other**
problems (leave-one-problem-out), and $\hat V$ on that problem is computed with them, so no value
is built from weights fitted to the trials it describes.

**The baseline.** Fitted the same held-out way: choice history + tower bias, no reward term. Its
choice probabilities are averaged into the same bins as the observed choices, so a solid curve that
rises more than its dashed baseline is a value effect beyond stickiness and port bias.

**Model comparison** (held out): reward history only; + choice history; + tower bias (the full
model). Scores are negative log likelihood per choice in nats, against $\log 3 \approx 1.0986$
for choosing at random.

**Bins.** The continuous value is cut into `N_BINS` bins of equal numbers of a mouse's trials (ranks,
with ties ordered at random, so a bin is never empty even when many values are exactly equal). Each
bin is placed on the x axis at its mean value (averaged over mice), so the axis is in logit units, and
a mean drawn from fewer than `MIN_MICE` mice is left out. Values that are discrete by nature
(task and believed magnitudes) are plotted at their exact values.

## 2. Synthetic data: Q-learning mice with known values

`simulate_cohort` makes agents of the value learner that the RL notebook and
`src/agents/drifting_simulation.py` use: a softmax over learned values $Q$ with inverse
temperature $\beta$, a stickiness bonus $\kappa$ on the previous choice, and a bias
`plus_bias` for the ports in the same row or column as the initiation port (the known
tower bias), after each choice $Q_a \leftarrow Q_a + \alpha\,(r - Q_a)$. They play the drifting
task (drift attempt rate 0.1, blocks of 30-35 trials, the most chosen arm drops to 0 at a block
change) on six different triples of towers, so every tower is offered in at least two problems and
its bias can be learned from the others. The start magnitudes of each problem are dealt to the ports
at random. The five parameter settings below are each run by two mice.

In [ ]:
synthetic = pipe.simulate_cohort(
    PARAMETER_SETS, mice_per_set=MICE_PER_SET, seed=SEED, **SYNTHETIC
)
print(f"{synthetic.trials['mouse'].nunique()} mice, {len(synthetic.trials):,} trials")
synthetic.parameters

### 2.1 The reward tail, and how many arms have a value

First the decay of the tail, chosen from `TAIL_GRID` by held-out likelihood (lower nats per choice
is better; the first row, with no tail, is for comparison). Then the fraction of (trial, arm) model
values that are exactly 0, without and with the tail.

In [ ]:
selection = pipe.select_tail_decay(synthetic.trials, TAIL_GRID, n_lags=N_LAGS, l2=L2)
analysis = selection.analysis
no_tail = pipe.analyse_trial_table(synthetic.trials, n_lags=N_LAGS, tail_decay=None, l2=L2)
print(f"chosen tail decay: {selection.decay}")
print(
    f"fraction of arm values exactly 0: {mvp.zero_fraction(no_tail.values):.3f} without the tail, "
    f"{mvp.zero_fraction(analysis.values):.3f} with it"
)
selection.table.round(4)

### 2.2 Fit, and kernels against what the generating rule implies

The Q-learner is not exactly a lagged-reward logit, so "recovered" needs a precise target. A reward
that is followed by $m$ further visits to the same arm adds $\beta\alpha(1-\alpha)^m$ to that
arm's utility (an unchosen arm keeps its value, a chosen arm forgets by a factor $1-\alpha$ per
visit). For a reward $k$ trials back, $m$ lies between 0 (never chosen again) and $k-1$ (chosen on
every trial since), so the reward's contribution at lag $k$ lies between

$$\beta\alpha(1-\alpha)^{k-1}\quad\text{and}\quad\beta\alpha .$$

The table gives these bounds as the scale to expect; the fitted weights need not fall inside them,
because the choice-history features and the tail share part of the effect. The **reference** (dashed
in the figure, `reference` in the table) is sharper: the weights of the logit's own features that
come closest, in cross-entropy, to each agent's *generating choice probabilities* (a logit fitted to
those probabilities as soft targets: what a logit would find with unlimited data from the agent,
no sampling noise). The tower biases of the generating rule are $0$ or `plus_bias` for each tower
(relative to the reference tower).

In [ ]:
reference = pipe.reference_weights(analysis, synthetic)
base = synthetic_dir / "Choice Logit Weights"
plots.plot_kernels(
    analysis.weights, base, reference=reference, reference_label="Generating Rule (Soft-Target Fit)"
)
show(base)
pipe.kernel_comparison(analysis, synthetic).round(3).query("lag <= 3")

**Reading the figure.** Thin solid lines are the fitted weights of each mouse, dashed lines the
reference of the same mouse, the thick line their means. Compare the solid and dashed means: the
fitted reward kernel is in the range of the reference, noisier than it, and the tower biases in the
right panel fall near the identity line.

### 2.3 Model comparison

Held-out negative log likelihood per choice: choice history and tower bias should help exactly for
the mice that have stickiness and bias.

In [ ]:
base = synthetic_dir / "Model Comparison"
plots.plot_model_comparison(analysis.comparison, base)
show(base)
gains = analysis.comparison.pivot(index="mouse", columns="model", values="gain_over_chance")[list(cv.MODELS)]
gains.join(synthetic.parameters.set_index("mouse")[["kappa", "plus_bias"]]).round(3)

### 2.4 Psychometrics: model value against the true values, with baselines

For each trial the three arms are ranked by value, and the curves show how often the mouse chose the
best and the second-ranked arm against the best value, the second value, and their difference.
Four rows, all computed on **the same trials of the same mice**:

1. the **model value** $\hat V$ (held out, from the logit),
2. the agents' **own Q-values** (the generating values),
3. the task's **true magnitudes** the ports offer,
4. the **believed values** (the last reward each arm paid; trials where every arm has been chosen
   once in the problem).

Every row has the dashed baseline: the choice-history + tower-bias model's probability of the same
outcome, binned the same way. Rows 3 and 4 are discrete (pulses), so they use exact values; rows 1
and 2 use quantile bins.

In [ ]:
q_values = pipe.generating_values(analysis, synthetic)
bases = dict(
    model=(None, "Model Value", "Logit", N_BINS),
    q=(q_values, "Agent Q-Value", "Pulses", N_BINS),
    task=(pipe.task_values(analysis), "Task Magnitude", "Pulses", None),
    believed=(pipe.believed_values(analysis), "Believed Value", "Pulses", None),
)
rows = []
for values, name, unit, n_bins in bases.values():
    curves, baselines = pipe.value_curves(analysis, n_bins, values, MIN_MICE)
    rows.append(
        plots.PsychometricRow(name, curves, plots.value_labels(name, unit, binned=n_bins is not None), baselines)
    )
base = synthetic_dir / "Psychometric Curves"
plots.plot_psychometric_rows(
    rows,
    "Psychometric Curves: Model, Q, Task and Believed Values",
    (
        "Rows rank the three arms by the model value (held-out logit), by the simulated agents' own "
        "Q-value, by the magnitude each port offers, and by the last reward each arm paid.",
        f"Rows 1 and 2 use per-mouse quantile bins ({N_BINS}); rows 3 and 4 use the exact values.",
    ),
    base,
    MIN_MICE,
)
show(base)

**How close is $\hat V$ to the generating $Q$?** Within a trial only differences between arms matter, so
compare the trial-centred values (each arm minus the trial's mean over the three arms): Spearman
correlation per mouse. The believed value is the baseline to beat: it needs no model at all. Both are
compared on the trials where every arm has a believed value.

In [ ]:
agreement = pipe.value_agreement(analysis, synthetic).set_index("mouse")
display(agreement.round(3).T)
print(
    "mean over mice: model value %.3f, believed value %.3f"
    % (agreement["model_value"].mean(), agreement["believed_value"].mean())
)

### 2.5 Stay/switch curves and switch targets on model values

P(stay) and P(switch) against the value of the arm chosen on the previous trial and of its best
alternative. A continuous value has no "chose the 0-value option" row, so the rows are whether the
previous choice was **rewarded** (a non-zero magnitude) or not. Left to right: the difference
$V(\text{chosen}) - V(\text{alt})$, $V(\text{chosen})$ and $V(\text{alt})$. The same figure for the
model value, the agents' Q-values and the believed values, each with its dashed baseline.

The **switch-target** figure asks where a switch goes. Only the pairs whose next choice is a switch
count; y is the fraction of those switches that go to the *best alternative* (the higher-valued of the
two arms not chosen before), against the gap $V(\text{best alt}) - V(\text{other arm})$ between those
two arms. A switch between two arms of equal value has no best alternative and is left out (counted in
the caption). One panel per reward row, a dotted line at 0.5 (a random pick), and the nuisance
baseline: the choice-history + tower-bias model's own P(best alternative | switch). A value that
guides the switch should rise above 0.5 with the gap, and above its baseline.

In [ ]:
for key, label, unit in (("model", "Model Value", "Logit"), ("q", "Agent Q-Value", "Pulses"), ("believed", "Believed Value", "Pulses")):
    values, _, _, n_bins = bases[key]
    pairs = pipe.value_pairs(analysis, values)
    curves, baselines = mvp.stay_switch_curves_by_reward(
        pairs, pool_trials=POOL_TRIALS, n_bins=n_bins, min_mice=MIN_MICE
    )
    base = synthetic_dir / f"Stay-Switch Curves ({label})"
    plots.plot_stay_switch_by_reward(
        curves, label, POOL_TRIALS, base,
        baselines_by_row=baselines, unit=unit, min_mice=MIN_MICE, binned=n_bins is not None,
    )
    show(base)

    target_curves, target_baselines = mvp.switch_target_curves_by_reward(pairs, n_bins=n_bins, min_mice=MIN_MICE)
    base = synthetic_dir / f"Switch Target Curves ({label})"
    plots.plot_switch_targets_by_reward(
        target_curves, label, mvp.tied_switches_by_reward(pairs), base,
        baselines_by_row=target_baselines, unit=unit, min_mice=MIN_MICE, binned=n_bins is not None,
    )
    show(base)

### 2.6 Negative control: an agent that ignores value

`Value-Blind Sticky` has $\beta=0$: its choices do not depend on $Q$ at all, only on repeating the
previous choice. `Value-Driven` is a Q-learner with the same learning rate. If the raw psychometric
of the model value were evidence of value use, both would show it. The number below is the rise of
P(best) from the lowest to the highest bin of the best value, once as drawn ("raw") and once as the
mean over mice of the observed-minus-baseline rise ("beyond baseline", `value_effect`): the value
effect with stickiness and tower bias subtracted.

In [ ]:
control_rows, control_effects = [], []
for name, parameters in CONTROL_SETS.items():
    cohort = pipe.simulate_cohort([parameters], mice_per_set=3, seed=SEED + 1, **SYNTHETIC)
    result = pipe.analyse_trial_table(cohort.trials, n_lags=N_LAGS, tail_decay=analysis.tail_decay, l2=L2)
    curves, baselines = pipe.value_curves(result, N_BINS, None, MIN_MICE)
    best = curves["best"]["best_val"].mean
    control_effects.append(
        dict(agent=name, raw_rise=best[-1] - best[0], rise_beyond_baseline=mvp.value_effect(curves, baselines))
    )
    control_rows.append(plots.PsychometricRow(name, curves, plots.value_labels("Model Value", "Logit"), baselines))
base = synthetic_dir / "Negative Control"
plots.plot_psychometric_rows(
    control_rows,
    "Negative Control: Value-Blind Sticky Agent and a Value-Driven Agent",
    ("Model-value psychometrics of three mice of each kind; the dashed baseline is the held-out "
     "choice-history + tower-bias model.",),
    base,
    MIN_MICE,
)
show(base)
pd.DataFrame(control_effects).set_index("agent").round(3)

For the value-blind agent the solid curve rises (recently chosen arms are the ones with a model value,
and it repeats them) but the dashed baseline rises with it, so the effect beyond the baseline is about
0. The value-driven agent keeps a positive effect beyond its baseline. This is a conservative number:
the held-out baseline absorbs part of what the value explains, since a value-driven agent also
repeats rewarded choices.

### 2.7 Negative controls: win-stay/lose-shift agents fitted to each mouse

The Q-learner of the sections above learns values, so its reward kernel, its value psychometrics and
its stay/switch curves are what a value learner looks like to the logit. To read the same figures for
the mice we also need agents that have **no value-guided switching at all**, run through exactly the
same pipeline. Three soft win-stay/lose-shift (WSLS) agents (`src/agents/null_agents.py`):

| Agent | P(stay) after the previous choice | Where a switch goes |
|---|---|---|
| **WSLS** | $\sigma(a_\text{win})$ after any reward, $\sigma(a_\text{lose})$ after none; magnitude ignored | uniform over the two other arms |
| **WSLS (Recency)** | the same | first other arm with $\sigma(w\,(h_1-h_2))$, $h$ = decayed count of past choices |
| **WSLS (Magnitude)** | $\sigma(a_\text{win}+b\,(m-1))$ after $m\ge1$ pulses, $\sigma(a_\text{lose})$ after none | as WSLS (Recency) |

None of them looks at what the *other* arms offer, so none can switch towards the better arm for
value reasons. WSLS (Magnitude) is the realistic one: magnitude grades *staying* (the mice's reward
kernel is graded), but the switch target stays a recency effect. $b=0$ recovers WSLS.
The first trial of a session, and the first after a dropped trial, is uniform and clears the trace,
as the logit's choice history is cleared. There is no tower bias: the agents have no port
preference (so the logit's baseline has less to absorb than for a mouse).

**A limit of the switch-target comparison.** The logit's choice-history baseline sees five choice
lags and no choice tail. The agents' trace decays by 0.9 per trial, so most of it lies beyond those
five lags and the baseline cannot absorb where a recency-guided switch goes: recency alone gives
above-baseline switch targets (section 2.7 shows this, with a fast-decaying trace that fits in five
lags as the control). The mice's perseveration is also long, so their own excess may be inflated by the
same blind spot. The switch regression of the next analysis controls for it with a long choice trace
and trials-since-chosen.

Each agent is **fitted to every mouse** (the stay logits in closed form or by a two-parameter
logistic fit, $w$ by a one-dimensional maximum likelihood), then **plays that mouse's own problems**:
the same towers, initiation port, starting magnitudes and number of trials per session, in a task
whose drift and block lengths are measured from the data. The magnitudes are generated by the task
rules, not replayed: a block change zeroes the arm the *agent* chose most, so the mouse's recorded
schedule belongs to the mouse's own choices. The simulated agents then go through the same logit
analysis as the mice. The likelihood of each agent is scored in-sample (two to four parameters
against thousands of choices; held-out scores are not needed), next to the logit's held-out score.

The helper below does all of that for one trial table and one reference analysis (the Q-learners
here, the mice in section 3.3), writing figures to a `negative-controls/` subfolder.

In [ ]:
def run_null_controls(trial_table, reference, reference_label, out_dir):
    """Fit the null agents to the mice of *reference*, play them, tabulate, draw and show."""
    comparison = nulls.compare_with_controls(
        trial_table, reference, reference_label, NULL_SEED,
        n_bins=N_BINS, min_mice=MIN_MICE, pool_trials=POOL_TRIALS, decay=NULL_DECAY,
    )
    null_plots.plot_comparison(comparison, out_dir)
    for kind, label in nulls.AGENT_LABELS.items():
        print(f"fitted parameters, {label}")
        display(nulls.parameters_table(comparison.controls.fits[kind]).round(3))
    print("negative log likelihood per choice (agents in-sample, logit held-out)")
    display(comparison.scores.pivot(index="mouse", columns="model", values="nats_per_choice").round(4))
    print("rise of P(best) from the lowest to the highest bin of the best value")
    display(comparison.effects.set_index("source").round(3))
    print("switch targets: observed minus choice baseline, by reward row")
    display(comparison.switch_excess.round(3))
    for name in ("Null Agent Fits", "Kernels by Source", "Psychometric Curves"):
        show(out_dir / name)
    for label in nulls.AGENT_LABELS.values():
        show(out_dir / f"{label} Stay-Switch Curves")
        show(out_dir / f"{label} Switch Target Curves")
    return comparison

**Run on the Q-learners.** Reference: the five parameter settings of section 2 (the analysis of
section 2.1, with its tail decay). Each null agent is fitted to a simulated Q-learner and plays
generated schedules matched to that mouse (its towers, initiation port, starting magnitudes and
trials per session, with the drift and block lengths measured from the data); read the null agents'
rows against the Q-learner's. The agents' likelihoods are in-sample, the logit's held out.

In [ ]:
null_synthetic = run_null_controls(synthetic.trials, analysis, "Q-Learner", synthetic_dir / "negative-controls")

**Reading the figures** (numbers from the run above, seed `SEED`).

- *Kernels by source.* The Q-learner's reward kernel is positive over all five lags (about 0.26,
  0.22, 0.13, 0.12, 0.09). The three win-stay/lose-shift agents have a lag-1 weight (WSLS and
  WSLS (Recency) about 0.33, WSLS (Magnitude) about 0.55) and about zero from lag 2 on.
- *Likelihood.* The agents' scores are in-sample (2 to 4 parameters), the logit's are held out. The
  logit has the lowest negative log likelihood for all ten mice, and within a mouse WSLS >= WSLS
  (Recency) >= WSLS (Magnitude) (the nested fits cannot do worse).
- *Psychometrics.* The rise of P(best) from the lowest to the highest bin beyond the choice baseline
  is 0.078 for the Q-learner and 0.056, 0.051 and 0.137 for WSLS, WSLS (Recency) and WSLS
  (Magnitude): it is not zero for the null agents. The baseline has no reward term, so staying after a
  reward (and, more, after a large reward) looks like a value effect. Read the Q-learner's and the
  mice's rise against these numbers, not against zero.
- *Switch targets.* The table gives observed minus baseline, averaged over bins (rewarded row,
  unrewarded row): Q-learner 0.096, 0.089; WSLS 0.006, 0.012; WSLS (Recency) 0.032, -0.004; WSLS
  (Magnitude) 0.041, 0.039. At the fitted switch weights (w about 0.2) the null agents sit close to
  the baseline. That is not a property of recency in general: with a strong recency weight (w = 1.5)
  and the default decay 0.9, the rewarded-row excess is about 0.30 in a test run
  (`tests/test_null_agents_pipeline.py`), and it is about 0 when the trace decays by 0.5 (it then
  fits in the baseline's five lags). The cause is the baseline's five choice lags without a choice tail, not
  value guidance. The same test shows the same pattern for WSLS (Magnitude): its excess is about
  0.30 at w = 1.5 and decay 0.9 and about 0 at w = 0 (graded staying alone) or at decay 0.5, so the
  small excess of the fitted agent is the recency effect and not graded staying. The switch regression
  of the next analysis controls for it with a long choice trace and trials-since-chosen.

### 2.8 Regression Extensions

Four questions about the choice logit, each asked of the same five sources so that the mice can be
read against a positive and three negative controls: the **Q-learners** above (they use reward
magnitudes and keep a value for every arm) and the three **win-stay/lose-shift agents** of 2.7
(no value for unchosen arms, magnitude only in WSLS (Magnitude)). Every number is held out:
a model is fitted on all but one problem of a mouse and scored on the problem it did not see, in
nats per choice (lower is better; chance is log 3 = 1.0986).

1. **Believed value.** The magnitude an arm last paid when the mouse chose it in the current
   problem, plus a **never-visited indicator** (1 before the arm's first visit, where the value is
   set to 0), so an unvisited arm is neither worth 0 nor dropped. Variants: (a) the full model,
   (b) full + believed value, (c) choice history + tower bias + believed value, without reward
   lags. The weights come from (b) fitted to all of a mouse's trials. If (b) is no better than (a),
   the believed value adds nothing to the reward lags.
2. **Binary against pulse.** The reward lags code pulses (the standard model). *Binary* codes 1 if
   rewarded and 0 otherwise, with the same lags and tail; *binary + pulse* has both. WSLS and WSLS
   (Recency) cannot see magnitude and should prefer binary; WSLS (Magnitude) and the Q-learner
   should prefer pulse. With fewer than 5 mice per source the figure shows mice individually, with
   no test.
3. **Unique and shared variance.** For tower bias, reward history and choice history: the unique
   variance of a group is the held-out cost of dropping it from the full model. The shared term is
   `[gain(reward + tower) + gain(choice + tower)] - gain(full)` with gains over the tower-only model,
   so unique reward + unique choice + shared = the full model's gain over the tower-only model
   (single-group models are in the tables too).
4. **Calibration.** Every (trial, arm) pair is a prediction; predictions are cut into quantile bins
   and the y axis is the **observed fraction of times that arm was chosen**, against the mean
   predicted probability. One panel for the full model, one for the model with choice history only.

Figures go to `regression-extensions/` next to the other figures of each section.

In [ ]:
def report_regression_extensions(sources, out_dir):
    """Tabulate the four analyses of each source, draw the figures, show them and the summaries."""
    tables = {label: ext.extension_tables(source, EXT_BINS) for label, source in sources.items()}
    ext_plots.plot_extensions(tables, out_dir)
    summaries = ext.source_summaries(tables)
    print("held-out gains in nats per choice (positive: better than the full pulse model, or, for the")
    print("believed-value rows, than the full model); p-values need at least 5 mice")
    display(summaries.gains.round(4))
    print("believed-value and never-visited weights (logit per pulse, logit)")
    display(summaries.weights.round(3))
    print("unique and shared variance, held-out nats per choice")
    display(summaries.partition.round(4))
    print("calibration: mean over bins of |observed - predicted|, and weighted by bin size")
    display(summaries.calibration.round(4))
    for name in ("Believed Value", "Binary vs Pulse", "Unique and Shared Variance"):
        show(out_dir / name)
    for label in sources:
        show(out_dir / f"Calibration ({label})")
    return tables, summaries

In [ ]:
ext_synthetic = ext.analyse_sources(
    ext.comparison_inputs("Q-Learner", synthetic.trials, null_synthetic), EXT_JOBS, EXT_CACHE_DIR
)
tables_synthetic, summaries_synthetic = report_regression_extensions(
    ext_synthetic, synthetic_dir / "regression-extensions"
)

**Reading the figures** (numbers from the run above, seed `SEED`; 10 mice per source; gains in held-out nats per choice, mean ± se, positive is better).

- *Believed value.* Adding it to the full model helps the Q-learner (+0.0125 ± 0.0023, all 10 mice, Wilcoxon p = 0.002) and nothing else (WSLS +0.0001, Recency +0.0004, Magnitude -0.0001, none significant). The Q-learner's believed-value weight is 0.30 ± 0.04 logit per pulse, the agents' 0.02 to 0.07. Believed value *instead of* the reward lags is no worse than the lags for the Q-learner (-0.0073 ± 0.0050, p = 0.23) but costs the null agents 0.006 to 0.025 (p = 0.002). The never-visited weight is noisy (Q-learner -0.26 ± 0.12, agents about 0 to +0.1).
- *Binary against pulse.* WSLS and WSLS (Recency) prefer binary in all 10 mice (+0.0168 and +0.0149); WSLS (Magnitude) and the Q-learner prefer pulse in all 10 (-0.0287 and -0.0270). Adding binary lags to pulse lags gains +0.0167 and +0.0141 for the two magnitude-blind agents and about +0.001 for the other two. The test separates the sources as expected.
- *Unique and shared variance.* Tower bias has no unique variance anywhere (-0.004 to +0.003). Unique reward is 0.077 for the Q-learner and 0.017, 0.017 and 0.057 for WSLS, Recency and Magnitude; unique choice is 0.005, 0.034, 0.039 and 0.011. The shared term dominates (0.36, 0.24, 0.24, 0.27): a rewarded choice is also a repeated choice, so reward and choice lags explain the same stays.
- *Calibration.* Mean |observed - predicted| across deciles of the full model is 0.014 (Q-learner), 0.027, 0.026 and 0.009 (WSLS, Recency, Magnitude); of the choice-history-only model 0.007, 0.008, 0.012 and 0.008. All are small, and the choice-only model is not better *predicting*: it is lower-variance (predictions stay between about 0.05 and 0.9).

## 3. Real data

The drifting problems of `COHORTS`, pooled as in the other all-cohort analyses (problems are keyed
`(cohort, number within the drifting type)`). The model is fitted per mouse and validated by leaving
one problem out. A mouse with a single drifting problem or fewer than 300 trials cannot be validated;
it is skipped with a warning and listed.

In [ ]:
problems = pipe.load_drifting_problems(COHORTS, DATA_ROOT)
trials = drifting_trial_table(problems)
trials.groupby(["mouse", "problem"]).size().unstack(fill_value=0)

In [ ]:
real_selection = pipe.select_tail_decay(trials, TAIL_GRID, n_lags=N_LAGS, l2=L2)
real = real_selection.analysis
real_no_tail = pipe.analyse_trial_table(trials, n_lags=N_LAGS, tail_decay=None, l2=L2)
print("skipped:", real.skipped or "none")
print(f"chosen tail decay: {real_selection.decay}")
print(
    f"fraction of arm values exactly 0: {mvp.zero_fraction(real_no_tail.values):.3f} without the tail, "
    f"{mvp.zero_fraction(real.values):.3f} with it"
)
print(f"{len(real.designs)} analysed mice: {sorted(real.designs)}")
base = real_dir / "Choice Logit Weights"
plots.plot_kernels(real.weights, base)
show(base)

Weights are those of the full model fitted to all of a mouse's trials. Compare the reward kernel with
the Q-learner's from section 2: a positive, decaying kernel is what a learner that updates its values from
the rewards it receives looks like. The choice kernel shows perseveration (and, as explained above, part of
the correction for repeatedly chosen arms).

In [ ]:
base = real_dir / "Model Comparison"
plots.plot_model_comparison(real.comparison, base)
show(base)
real.comparison.pivot(index="mouse", columns="model", values="nats_per_choice")[list(cv.MODELS)].round(4)

### 3.1 Psychometrics: model value next to the task and believed values

Row 1 ranks the arms by the held-out model value (quantile bins per mouse); row 2 by the magnitudes
the ports offer; row 3 by the believed value (what an arm paid the last time the mouse chose it;
trials once all three arms of the problem have been chosen). All rows use the **same mice and
problems** as the model value, and each has the dashed baseline.

In [ ]:
real_bases = dict(
    model=(None, "Model Value", "Logit", N_BINS),
    task=(pipe.task_values(real), "Task Magnitude", "Pulses", None),
    believed=(pipe.believed_values(real), "Believed Value", "Pulses", None),
)
rows = []
for values, name, unit, n_bins in real_bases.values():
    curves, baselines = pipe.value_curves(real, n_bins, values, MIN_MICE)
    rows.append(
        plots.PsychometricRow(name, curves, plots.value_labels(name, unit, binned=n_bins is not None), baselines)
    )
base = real_dir / "Psychometric Curves"
plots.plot_psychometric_rows(
    rows,
    "Psychometric Curves: Model, Task and Believed Values",
    (
        "Rows rank the arms by the held-out model value (per-mouse quantile bins), by the magnitude "
        "each port offers, and by the last reward each arm paid (exact values).",
        "Row 3 counts trials once all three arms of the problem have been chosen.",
    ),
    base,
    MIN_MICE,
)
show(base)

### 3.2 Stay/switch curves: model value next to believed value

Rows by whether the previous choice was rewarded. First the model value, then the believed value, each
with its dashed baseline; each is followed by its switch-target figure (best alternative against the
gap to the other unchosen arm, switches only).

In [ ]:
for key, label, unit in (("model", "Model Value", "Logit"), ("believed", "Believed Value", "Pulses")):
    values, _, _, n_bins = real_bases[key]
    pairs = pipe.value_pairs(real, values)
    curves, baselines = mvp.stay_switch_curves_by_reward(
        pairs, pool_trials=POOL_TRIALS, n_bins=n_bins, min_mice=MIN_MICE
    )
    base = real_dir / f"Stay-Switch Curves ({label})"
    plots.plot_stay_switch_by_reward(
        curves, label, POOL_TRIALS, base,
        baselines_by_row=baselines, unit=unit, min_mice=MIN_MICE, binned=n_bins is not None,
    )
    show(base)

    target_curves, target_baselines = mvp.switch_target_curves_by_reward(pairs, n_bins=n_bins, min_mice=MIN_MICE)
    base = real_dir / f"Switch Target Curves ({label})"
    plots.plot_switch_targets_by_reward(
        target_curves, label, mvp.tied_switches_by_reward(pairs), base,
        baselines_by_row=target_baselines, unit=unit, min_mice=MIN_MICE, binned=n_bins is not None,
    )
    show(base)

### 3.3 Negative controls on the mice

The same three win-stay/lose-shift agents (section 2.7), fitted to each analysed mouse and played on
that mouse's own problems, then run through the logit: kernels, model comparison, psychometrics with
the baseline, and stay/switch and switch-target curves. Compare each row with the mice's figures above.

In [ ]:
null_real = run_null_controls(trials, real, "Mice", real_dir / "negative-controls")

**Reading the figures** (one mouse, MY_175_L, so there are no se bands and nothing here is a
group statement). Fitted: WSLS (Magnitude) has `a_win` 0.43, `b` 0.29 per extra pulse and `w` 0.13;
the observed P(stay) rises from 0.49 after 0 pulses to 0.79 after 4, and WSLS (Magnitude) follows
it (the other two agents are flat from 1 pulse on). The agents' likelihoods are in-sample, the
logit's held out: negative log likelihood per choice 0.866 (WSLS), 0.858 (Recency), 0.851
(Magnitude) and 0.771 (logit). The mouse's rise of P(best) beyond the baseline is 0.148 against
0.051, 0.050 and 0.124 for the agents. Its switch-target excess is -0.046 (rewarded) and -0.012
(unrewarded), the agents' lie between -0.03 and 0.04: on this mouse the switch targets show nothing
beyond the baseline, and the baseline's five-lag blind spot (see 2.7) applies to the mouse as it does
to the agents.

### 3.4 Regression Extensions

The four analyses of 2.8 on the mice, next to the controls: the **Q-learners** of section 2 (the
analysis of 2.8, reused) and the **win-stay/lose-shift agents** fitted to each mouse of section 3.3
(the same simulated agents, reused). Read each mouse's gains, weights and calibration against the
controls; the mice are as few as the staged cohorts hold.

In [ ]:
ext_real = ext.analyse_sources(ext.comparison_inputs("Mice", trials, null_real), EXT_JOBS, EXT_CACHE_DIR)
real_sources = {
    "Mice": ext_real["Mice"],
    "Q-Learner": ext_synthetic["Q-Learner"],
    **{label: source for label, source in ext_real.items() if label != "Mice"},
}
tables_real, summaries_real = report_regression_extensions(real_sources, real_dir / "regression-extensions")

**Reading the figures** (one mouse, MY_175_L, so no se bars and no test; the Q-learner column is the 10 simulated mice of 2.8). Believed value adds nothing to the full model (-0.0005 nats per choice; weight 0.011 logit per pulse), and replacing the reward lags with it costs 0.0126, more than for the Q-learner (-0.0073 gain, i.e. 0.0073 cost) and about like WSLS (Magnitude) (0.0088 for the fitted agent). Its never-visited weight is -0.78 logit: an arm not yet visited in a problem is chosen less than an arm that last paid 0. Pulse is slightly preferred to binary (-0.0021) and adding binary lags to the pulse lags does not help (-0.0024), as for the Q-learner and WSLS (Magnitude) rather than the two magnitude-blind agents (+0.0072, +0.0049). Dropping the tower bias *improves* held-out fit (unique tower -0.020): with few repeated towers across a mouse's problems the fold's tower weights do not transfer. Unique reward is -0.0004, unique choice 0.039 and shared 0.323; the agents fitted to this mouse have unique reward 0.010 to 0.014 and shared 0.18 to 0.20. Calibration: the full model's mean |observed - predicted| is 0.026 and the choice-history-only model's 0.046 (the agents: 0.012 to 0.017 and 0.008 to 0.012). One mouse licenses no group statement; these are the numbers the cohort run will fill in.

### Notes and limits

- $\hat V$ is a *linear, finite-memory* summary of a mouse's reward history ($K$ = `N_LAGS` trials plus
  the decayed tail), chosen because it can be read straight from choices. It is not a Q-learner's value
  (flatter kernel; part of the Q-learner's forgetting sits in the choice weights); a real mouse need not
  be a Q-learner.
- The model value is in logits. Quantile bins place each bin at its mean value, so the x axis has the
  value's units, but the scale differs between mice (each mouse has its own weights); the x of a
  curve is the mean over mice of the bin means.
- Tower biases of towers that a fold's training problems never offered stay at 0 (the penalty), so
  the model comparison's tower step only helps where towers repeat across a mouse's problems.
- Real-data figures are as good as the number of mice. The staged cohort snapshots used here hold few
  drifting mice, and a mean from fewer than `MIN_MICE` mice is only drawn when the data have fewer
  mice than that in total; with one mouse the curves have no standard error.